# Two-Stage Retrieval with scikit-rec

scikit-rec is a **ranking library**. The retriever narrows a large item catalog to a
manageable candidate set; the ranker then scores those candidates precisely.

This notebook shows the three built-in retrievers and the `item_subset` external retrieval seam.

In [ ]:
import numpy as np
import pandas as pd

from skrec.constants import ITEM_ID_NAME, LABEL_NAME, USER_ID_NAME
from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.embedding.matrix_factorization_estimator import MatrixFactorizationEstimator
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.retriever import ContentBasedRetriever, EmbeddingRetriever, PopularityRetriever
from skrec.scorer.universal import UniversalScorer

np.random.seed(42)

## Synthetic Dataset

We create a small catalog of 20 items with two numeric features (price, avg_rating),
and 50 users each with 5 interactions.

In [ ]:
N_USERS = 50
N_ITEMS = 20
N_INTERACTIONS = 250

item_ids = [f"item_{i:03d}" for i in range(N_ITEMS)]
items_df = pd.DataFrame(
    {
        ITEM_ID_NAME: item_ids,
        "price": np.round(np.random.uniform(5, 100, N_ITEMS), 2),
        "avg_rating": np.round(np.random.uniform(1, 5, N_ITEMS), 1),
    }
)

user_ids = [f"user_{i:03d}" for i in range(N_USERS)]
interactions_df = pd.DataFrame(
    {
        USER_ID_NAME: np.random.choice(user_ids, N_INTERACTIONS),
        ITEM_ID_NAME: np.random.choice(item_ids, N_INTERACTIONS),
        LABEL_NAME: np.random.choice([0.0, 1.0], N_INTERACTIONS, p=[0.3, 0.7]),
    }
)
interactions_df = interactions_df.drop_duplicates([USER_ID_NAME, ITEM_ID_NAME]).reset_index(drop=True)

print(f"Items:        {len(items_df)}")
print(f"Users:        {len(interactions_df[USER_ID_NAME].unique())}")
print(f"Interactions: {len(interactions_df)}")
items_df.head(3)

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/retrieval")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Datasets expect file paths — write synthetic data to a stable directory
_interactions_path = str(DATA_DIR / "interactions.csv")
_items_path = str(DATA_DIR / "items.csv")

if not Path(_interactions_path).exists():
    interactions_df.to_csv(_interactions_path, index=False)
if not Path(_items_path).exists():
    items_df.to_csv(_items_path, index=False)

interactions_ds = InteractionsDataset(data_location=_interactions_path)
items_ds = ItemsDataset(data_location=_items_path)

## 1. EmbeddingRetriever — Personalized latent-space retrieval

`EmbeddingRetriever` uses the learned user and item factor matrices from a trained
embedding model (MF, NCF, Two-Tower). It computes `U @ I.T` to retrieve the
top-K candidates for each user, then the ranker re-scores them precisely.

**Best for**: warm users with interaction history, stable catalogs.  
**Not for**: new users or items added after training.

In [ ]:
recommender_emb = RankingRecommender(
    scorer=UniversalScorer(estimator=MatrixFactorizationEstimator(n_factors=16, epochs=20, random_state=42)),
    # top_k=5 here is the retrieval pool size — the ranker re-scores these 5 candidates
    # and returns the final top_k passed to recommend() (3 here).
    # In production with a large catalog, set retriever.top_k to 10–20× your final top_k
    # (e.g. retrieve 100–200 candidates, rank to top-10) to avoid precision loss from
    # truncating the pool too aggressively. The synthetic catalog here only has 20 items
    # so a small pool is fine for demonstration purposes.
    retriever=EmbeddingRetriever(top_k=5),
)
recommender_emb.train(interactions_ds=interactions_ds, items_ds=items_ds)

# For embedding-based scorers, pass only USER_ID — the model looks up the learned user vector
query_user = pd.DataFrame({USER_ID_NAME: ["user_000"]})
recs = recommender_emb.recommend(interactions=query_user, top_k=3)
print("EmbeddingRetriever recommendations for user_000:", recs[0])

## 2. ContentBasedRetriever — Cold-start & new items

`ContentBasedRetriever` uses item feature vectors (price, avg_rating, etc.) to
build a cosine-similarity index. The user profile is the mean of feature vectors
of their interacted items.

**Key differentiator**: items added to the catalog *after* training are immediately
retrievable — no retraining needed.

**Best for**: cold-start users, frequently changing catalogs.  
**Not for**: ID-only catalogs with no numeric item features.

In [ ]:
recommender_cb = RankingRecommender(
    scorer=UniversalScorer(estimator=MatrixFactorizationEstimator(n_factors=16, epochs=20, random_state=42)),
    retriever=ContentBasedRetriever(
        top_k=5,
        feature_columns=["price", "avg_rating"],
        weight_by_outcome=True,
    ),
)
recommender_cb.train(interactions_ds=interactions_ds, items_ds=items_ds)

# Add 3 new items to the catalog AFTER training — EmbeddingRetriever would miss these.
# For ContentBasedRetriever, rebuild its index with the extended items DataFrame.
new_items = pd.DataFrame(
    {
        ITEM_ID_NAME: ["item_NEW_A", "item_NEW_B", "item_NEW_C"],
        "price": [12.5, 88.0, 45.0],
        "avg_rating": [4.8, 2.1, 3.9],
    }
)
extended_items_df = pd.concat([items_df, new_items], ignore_index=True)

# Rebuild the retriever index with the extended catalog (no model retraining needed)
recommender_cb.retriever.build_index(
    interactions=interactions_df,
    items=extended_items_df,
)

candidates = recommender_cb.retriever.retrieve(["user_001"], top_k=5)
print("ContentBasedRetriever candidates for user_001:", candidates["user_001"])
print("(new items are immediately included in the index)")

# NOTE: we call retrieve() directly here rather than recommend() to highlight that
# new items (item_NEW_*) appear in the retrieval pool. Calling recommend() on top
# of these candidates would work correctly with a feature-based scorer (XGBoost,
# LightGBM), which scores items using the same price/avg_rating features the
# retriever was built on — new items have those features, so ranking works end-to-end.
# With the MF scorer used here, recommend() would fail on new item IDs because MF
# is purely ID-based and has no representation for items it did not see during training.
# ContentBasedRetriever is therefore most naturally paired with feature-based scorers.

## 3. PopularityRetriever — Non-personalized fallback

`PopularityRetriever` returns the globally most-interacted items for every user.
Works with **any** estimator including XGBoost — no embedding space required.

**Best for**: getting started, cold-start scenarios, non-personalized use cases.  
**Not for**: personalized recommendations (all users get the same candidate set).

In [ ]:
recommender_pop = RankingRecommender(
    scorer=UniversalScorer(estimator=MatrixFactorizationEstimator(n_factors=16, epochs=20, random_state=42)),
    retriever=PopularityRetriever(top_k=5),
)
recommender_pop.train(interactions_ds=interactions_ds, items_ds=items_ds)

for uid in ["user_000", "user_001", "user_002"]:
    candidates = recommender_pop.retriever.retrieve([uid], top_k=5)
    print(f"{uid}: {candidates[uid]}")
print("\nAll users get the same popular candidates — the ranker adds personalization.")

## 4. External retrieval via item_subset

For large catalogs, bring your own retrieval system (Elasticsearch, FAISS, rules).
Pass the candidates directly via `set_item_subset()` — no retriever attached to the recommender.

This is the right choice when:
- You already have a retrieval system in production.
- Retrieval logic is driven by business rules (only in-stock items, geo-filtered, etc.).

In [ ]:
recommender_base = RankingRecommender(
    scorer=UniversalScorer(estimator=MatrixFactorizationEstimator(n_factors=16, epochs=20, random_state=42)),
    # No retriever — we pass item_subset manually
)
recommender_base.train(interactions_ds=interactions_ds, items_ds=items_ds)

# Simulate external retrieval (e.g. Elasticsearch returning 5 candidates)
external_candidates = ["item_000", "item_005", "item_010", "item_015", "item_019"]

# For embedding-based scorers, pass only USER_ID
query_user = pd.DataFrame({USER_ID_NAME: ["user_003"]})

recommender_base.set_item_subset(external_candidates)
recs = recommender_base.recommend(
    interactions=query_user,
    top_k=3,
)
recommender_base.clear_item_subset()

print("Ranked from external candidates:", recs[0])
print("All results are within the externally retrieved set:", all(r in external_candidates for r in recs[0]))

## 5. Full two-stage pipeline: retrieve() then recommend()

The previous cells show `retrieve()` and `recommend()` separately. Here we call both
explicitly for the same user so you can see exactly what each stage does and verify
that the final recommendations always come from the retrieved candidate pool.

In [ ]:
# Step 1: retrieve candidates
uid = "user_005"
retrieved = recommender_emb.retriever.retrieve([uid], top_k=5)
print(f"Retrieved candidates for {uid}:")
for rank, item in enumerate(retrieved[uid], 1):
    print(f"  {rank}. {item}")

# Step 2: rank candidates and return top-3
# recommend() runs retrieve() internally and scores only those candidates.
# We call retrieve() above just to make the intermediate step visible.
query = pd.DataFrame({USER_ID_NAME: [uid]})
recs = recommender_emb.recommend(interactions=query, top_k=3)
print()
print(f"Final top-3 recommendations for {uid}:")
for rank, item in enumerate(recs[0], 1):
    print(f"  {rank}. {item}")

# Verify containment: every recommendation must come from the retrieved pool
candidate_set = {str(c) for c in retrieved[uid]}
assert all(str(r) in candidate_set for r in recs[0]), "recommendation outside candidate pool!"
print()
print("All recommendations come from the retrieved candidate pool.")

## 6. Why two stages?

| Stage | Goal | Typical scale |
|---|---|---|
| **Retrieval** | Narrow catalog → candidate set | 1M items → top 500 |
| **Ranking** | Score candidates precisely | 500 items → top 10 |

Ranking models (XGBoost, DeepFM, NCF) are expensive — running them over 1M items
per user is infeasible. Retrieval makes ranking tractable.

scikit-rec sits in the **ranking stage**. The three built-in retrievers cover common
use cases out of the box; `item_subset` connects to any external retrieval system.

### Choosing a retriever

| | Personalized | Needs training | New items | New users | XGBoost |
|---|---|---|---|---|---|
| `EmbeddingRetriever` | ✓ | ✓ | ✗ | ✗ | ✗ |
| `ContentBasedRetriever` | Partial | ✗ | ✓ | Partial | ✓ |
| `PopularityRetriever` | ✗ | ✗ | ✓ | ✓ | ✓ |
| External `item_subset` | You decide | — | — | — | ✓ |